# GPU-Accelerated Edge Detection with TensorFlow

This notebook applies a **Sobel edge-detection filter** to an image by running a 2D convolution on the GPU.

**Pipeline:**
1. Verify the GPU is visible to TensorFlow.
2. Upload and pre-process the image (grayscale + resize + normalize).
3. Build the Sobel-X and Sobel-Y kernels.
4. Run the convolution on `/GPU:0` and combine the gradients into edge magnitude.
5. Display, save, and log the result.

Before running, make sure the runtime is set to GPU: **Runtime → Change runtime type → GPU**.

## 1. Environment check

TensorFlow can silently fall back to the CPU if the GPU runtime is misconfigured. The first thing we do is confirm a GPU is actually attached — if this list is empty, the rest of the notebook will run on the CPU.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs detected      : {len(gpus)}")
for gpu in gpus:
    print(f"  - {gpu.name} ({gpu.device_type})")

if not gpus:
    print("\n[!] No GPU found. Switch runtime to GPU: Runtime > Change runtime type.")

## 2. Upload and preprocess the image

We convert to grayscale (single channel) so the Sobel kernel has a straightforward 2-D signal to operate on, resize to a fixed shape for predictable timing, and normalize pixel values to `[0, 1]` so the convolution output stays in a numerically friendly range.

In [ ]:
import numpy as np
from PIL import Image
from google.colab import files

IMAGE_SIZE = (256, 256)

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded. Re-run this cell and select an image.")

filename = next(iter(uploaded))
print(f"Loaded: {filename}")

img = Image.open(filename).convert('L').resize(IMAGE_SIZE)
img_np = np.asarray(img, dtype=np.float32) / 255.0

# tf.nn.conv2d expects shape [batch, height, width, channels].
img_tf = tf.convert_to_tensor(img_np[None, ..., None], dtype=tf.float32)

print(f"Tensor shape : {img_tf.shape}  (batch, H, W, channels)")
print(f"Pixel range  : [{img_np.min():.3f}, {img_np.max():.3f}]")

## 3. Build the Sobel kernels

Sobel approximates the image-intensity gradient. `Gx` responds to **horizontal** intensity changes (vertical edges); `Gy` responds to **vertical** intensity changes (horizontal edges). Combining them via `sqrt(Gx² + Gy²)` gives the total **edge magnitude**, which is what the human eye intuitively recognizes as "an edge."

The kernels are reshaped to `[kernel_h, kernel_w, in_channels, out_channels]` — the layout `tf.nn.conv2d` requires.

In [ ]:
sobel_x = tf.constant([[-1.0, 0.0, 1.0],
                       [-2.0, 0.0, 2.0],
                       [-1.0, 0.0, 1.0]], dtype=tf.float32)

sobel_y = tf.constant([[-1.0, -2.0, -1.0],
                       [ 0.0,  0.0,  0.0],
                       [ 1.0,  2.0,  1.0]], dtype=tf.float32)

sobel_x = tf.reshape(sobel_x, [3, 3, 1, 1])
sobel_y = tf.reshape(sobel_y, [3, 3, 1, 1])

## 4. Run the convolution on the GPU

Wrapping the op in `@tf.function` lets TensorFlow trace it into a graph the first time it is called, which removes Python overhead on subsequent calls. The `with tf.device('/GPU:0')` block forces placement onto the GPU — if no GPU is available this raises `RuntimeError` instead of silently using the CPU.

We also do a small warm-up call so the timed run reflects steady-state GPU performance rather than one-time CUDA / kernel-compilation cost.

In [ ]:
import time

@tf.function
def sobel_edges(x):
    gx = tf.nn.conv2d(x, sobel_x, strides=[1, 1, 1, 1], padding='SAME')
    gy = tf.nn.conv2d(x, sobel_y, strides=[1, 1, 1, 1], padding='SAME')
    return tf.sqrt(tf.square(gx) + tf.square(gy))

with tf.device('/GPU:0'):
    _ = sobel_edges(img_tf)        # warm-up: builds the graph and JITs the kernel
    start = time.perf_counter()
    edges_tf = sobel_edges(img_tf)
    _ = edges_tf.numpy()           # forces the GPU to finish before we stop the clock
    elapsed_ms = (time.perf_counter() - start) * 1000

print(f"GPU convolution time: {elapsed_ms:.3f} ms")

## 5. Post-process and display

The raw gradient magnitude can exceed 1.0, so we normalize by its own max before scaling to 8-bit. This keeps the output bright and well-contrasted regardless of the input image.

In [ ]:
import matplotlib.pyplot as plt

edges_np = tf.squeeze(edges_tf).numpy()
edges_np = edges_np / (edges_np.max() + 1e-8)            # normalize to [0, 1]
edges_uint8 = np.clip(edges_np * 255.0, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_np, cmap='gray')
axes[0].set_title("Original (grayscale)")
axes[0].axis('off')

axes[1].imshow(edges_uint8, cmap='gray')
axes[1].set_title("Sobel edge magnitude (GPU)")
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 6. Save outputs and download

We save the edge-detected image as a PNG and write a `log.txt` that records the TensorFlow version, the GPU that ran the convolution, the input shape, and the measured time — so the artifact itself is proof of GPU execution rather than just a screenshot.

In [ ]:
from datetime import datetime

OUTPUT_IMAGE = "gpu_output_tf.png"
LOG_FILE = "log.txt"

Image.fromarray(edges_uint8).save(OUTPUT_IMAGE)

with open(LOG_FILE, "w") as f:
    f.write("GPU Edge Detection - Execution Log\n")
    f.write("==================================\n")
    f.write(f"Timestamp        : {datetime.utcnow().isoformat()}Z\n")
    f.write(f"TensorFlow       : {tf.__version__}\n")
    f.write(f"Detected GPUs    : {tf.config.list_physical_devices('GPU')}\n")
    f.write(f"Input file       : {filename}\n")
    f.write(f"Input shape      : {tuple(img_tf.shape)}\n")
    f.write(f"GPU conv time    : {elapsed_ms:.3f} ms\n")
    f.write(f"Output file      : {OUTPUT_IMAGE}\n")

print(f"Saved {OUTPUT_IMAGE} and {LOG_FILE}")

files.download(OUTPUT_IMAGE)
files.download(LOG_FILE)